# Notebook 08 — Lint pass + archive with sync backlink rewrite

**Purpose:** Build the §10 Scenario D + H flows. Two demanding tasks share one notebook because they share a substrate (a wiki walker that yields validated pages):

- **Lint** — Opus 4.7 reasons across decisions and concepts to surface contradictions; pure-Python detectors find staleness and orphans.
- **Archive** — when a page moves to `archived/`, every backlink — body `[[wikilinks]]` and `related[]` frontmatter — is rewritten in one atomic pass.

**Exam relevance:** Architecture Patterns (lint as scheduled batch flow + multi-file atomic mutation), Prompt Engineering (deep CoT for Opus contradiction detection), Models & Capabilities (Opus 4.7 vs Sonnet 4.6 trade-offs).
**Design refs:** §10 Scenario D (lint pass), §10 Scenario H (archive with sync backlink rewrite), §12 failure modes, §16.1 Q4.
**Depends on:** NB 01 (page schemas), NB 02 (ingest), NB 05 (cross-source fixtures), NB 06 (Tool Use mechanics), NB 07 (orchestrator + QA).

**Wiki state:** the 5 committed `SourcePage` fixtures plus the new `knowledge/` tree (2 entities, 3 decisions including the deliberately-contradicting Apollo Q2/Q3 pair, 2 concepts) under `data/poc-wiki/`.

**Substrate insight:** lint reads the wiki; archive reads-and-rewrites it. One `walk_wiki` primitive feeds both. Keeping it separate from each agent is what lets the rewriter remain atomic — every patch is computed in memory before any file is written, and every patched page is re-validated against its Pydantic schema before any I/O happens.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import json
import time
from datetime import date
from pathlib import Path

from dotenv import load_dotenv
from anthropic import Anthropic
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.tree import Tree
from rich.markdown import Markdown
from IPython.display import Markdown as IPyMarkdown

from engine.utils.wiki_walker import walk_wiki
from engine.utils.backlink_rewrite import (
    compute_archive_patchset,
    apply_archive_patchset,
    render_patchset_diff,
    copy_wiki_to,
    all_referenced_wikilinks,
)
from engine.utils.cost_tracker import estimate_cost_usd
from engine.utils.api_compat import temperature_kwargs
from engine.agents.lint import (
    lint_wiki,
    detect_stale,
    detect_orphans,
    detect_contradictions,
    format_pages_block,
    format_lint_report_md,
    Contradiction,
    LintReport,
)
from engine.models.pages import ArchivedReason, PageType, PageStatus
from engine.models.wiki_config import MarginaliaConfig
from engine.prompts import load_prompt

console = Console()
TODAY = date(2026, 5, 3)
WIKI = Path("data/poc-wiki").resolve()


In [ ]:
load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not in env"

client = Anthropic()
config = MarginaliaConfig.load(WIKI)

# Smoke: walk the wiki, expect 12 pages.
pages = list(walk_wiki(WIKI))
print(f"walked {len(pages)} pages from {WIKI}")
print(f"types: {sorted({p.page_type.value for p in pages})}")


## Part A — `wiki_walker`: the iteration primitive

Both lint and archive need to walk every page in the wiki. Putting the
loop in one place avoids two slightly-different "for every page" loops
drifting apart.

`walk_wiki(root)` yields a `WikiPage` per validated `.md` page:

- `path` — absolute filesystem path (for read/write).
- `wikilink` — root-relative, extension-less form used in `[[...]]`
  references (e.g. `knowledge/decisions/apollo-q2-ship`).
- `page_type` — the `PageType` enum value.
- `page` — the validated Pydantic model.
- `body` — the markdown body (post-frontmatter).

Skip rules: `raw/` (inbox), `sources/raw/` (durable archive of
originals), hidden dirs, and any `.md` whose frontmatter `type:`
doesn't match a `PageType` (covers `purpose.md`, `AGENTS.md`,
`README.md` without listing them by name).


In [ ]:
t = Table(title=f"All pages under {WIKI.name}/ (n={len(pages)})")
t.add_column("type")
t.add_column("wikilink")
t.add_column("status")
t.add_column("last_synced")
t.add_column("related", justify="right")

for p in pages:
    t.add_row(
        p.page_type.value,
        p.wikilink,
        p.page.status.value,
        p.page.last_synced.isoformat(),
        str(len(p.page.related)),
    )

console.print(t)


## Part B — Fixture tour: the seeded knowledge graph

The 7 hand-written `knowledge/` pages were added so this notebook has
something real to lint and archive. Highlights of the seeded state:

| Page | Why it's seeded that way |
|---|---|
| `knowledge/decisions/apollo-q2-ship` | `status: active`, `last_synced: 2026-04-15` (18 days stale → triggers staleness) |
| `knowledge/decisions/apollo-q3-postpone` | `status: active`, **no `supersedes`** → contradicts q2-ship, lint must catch |
| `knowledge/decisions/engine-model-ladder` | Sane, consistent, recent — control case |
| `knowledge/concepts/ingest-pipeline` | Cross-references entities + decisions → exercises orphan detection negatively |
| `knowledge/entities/{apollo-project,marginalia-engine}` | Hub pages; referenced by decisions and sources |

Both Apollo decisions are **deliberately left with `contradicts: []`** —
the whole point of the contradiction-detection demo is that lint
finds the conflict blind.


In [ ]:
# Show the two contradicting Apollo decisions side-by-side.
q2 = next(p for p in pages if p.wikilink == "knowledge/decisions/apollo-q2-ship")
q3 = next(p for p in pages if p.wikilink == "knowledge/decisions/apollo-q3-postpone")

console.print(Panel(
    f"[bold]Title:[/] {q2.page.title}\n"
    f"[bold]Status:[/] {q2.page.status.value}\n"
    f"[bold]Last synced:[/] {q2.page.last_synced} (today is {TODAY})\n"
    f"[bold]Body:[/]\n{q2.body.strip()}",
    title=q2.wikilink,
    border_style="green",
))
console.print(Panel(
    f"[bold]Title:[/] {q3.page.title}\n"
    f"[bold]Status:[/] {q3.page.status.value}\n"
    f"[bold]Last synced:[/] {q3.page.last_synced}\n"
    f"[bold]Body:[/]\n{q3.body.strip()}",
    title=q3.wikilink,
    border_style="red",
))
print("\nNote: both decisions are status=active with empty `contradicts[]`.")
print("Lint must surface the conflict from the prose alone.")


## Part C — Stale detection (pure Python)

No LLM call. For each page, compute `(today - last_synced).days`. The
default threshold is 14 days (per design §10 Scenario D). The seeded
`apollo-q2-ship` decision has `last_synced: 2026-04-15`, so against
today (`2026-05-03`) it's 18 days stale and should appear.

Several committed source pages have `last_synced` in 2025 (artifacts of
NB 02's ingest fixture build) — those will also surface as stale,
which is honest output: in production they'd be the first targets for
re-ingest.


In [ ]:
stale = detect_stale(pages, threshold_days=14, today=TODAY)

t = Table(title=f"Stale pages (threshold=14d, today={TODAY.isoformat()})")
t.add_column("wikilink")
t.add_column("type")
t.add_column("last_synced")
t.add_column("days_stale", justify="right")
for s in stale:
    t.add_row(s.wikilink, s.page_type.value, s.last_synced.isoformat(), str(s.days_stale))

console.print(t)
print(f"\nFound {len(stale)} stale page(s).")
print("`apollo-q2-ship` is the deliberately-seeded target; the 2025-dated sources are NB 02 fixture artifacts.")


## Part D — Contradiction detection (Opus 4.7)

This is the cert-valuable cell. Opus 4.7 walks every pair of decisions
and concepts and surfaces contradictions with quoted evidence. The
prompt requires `<thinking>` CoT before the structured JSON output.

Three things make this work:

1. **Scope.** Only decisions + concepts go in. Sources are facts;
   entities are hubs. Including them dilutes signal.
2. **CoT.** Opus's reasoning budget is the whole point of using it
   here; a `<thinking>` block in the prompt forces explicit pairwise
   comparison.
3. **Severity grading.** "high" means directly opposed factual
   claims. The Apollo Q2/Q3 case is the canonical "high" — same
   project, same decision type, opposite quarters.

We'll call Opus once, then A/B against Sonnet 4.6 to demonstrate the
model-choice rationale that lands in the receipts.


In [ ]:
# Show the formatted block that goes to the model.
block = format_pages_block(pages)
print(f"feeding {block.count('<page id=')} pages to Opus (decisions + concepts only)")
print(f"block size: {len(block):,} chars")
print("---")
print(block[:600], "... (truncated)")
print("---")

prompt = load_prompt("lint")
print(f"\nPrompt {prompt.name} v{prompt.version}, role={prompt.role}, model={prompt.model}")


In [ ]:
# Opus run — the canonical lint call.
t0 = time.perf_counter()
opus_contradictions, opus_in, opus_out = detect_contradictions(
    pages, client=client, model="claude-opus-4-7"
)
opus_wall = time.perf_counter() - t0
opus_cost = estimate_cost_usd("claude-opus-4-7", opus_in, opus_out)

t = Table(title="Opus 4.7 contradiction findings")
t.add_column("#")
t.add_column("page_a")
t.add_column("page_b")
t.add_column("severity")
t.add_column("subject")
for i, c in enumerate(opus_contradictions, 1):
    t.add_row(str(i), c.page_a, c.page_b, c.severity, c.subject)
console.print(t)

# Validate the seeded contradiction was caught.
caught = any(
    {c.page_a, c.page_b} == {
        "knowledge/decisions/apollo-q2-ship",
        "knowledge/decisions/apollo-q3-postpone",
    }
    for c in opus_contradictions
)
assert caught, "Opus failed to flag the seeded apollo-q2-ship vs apollo-q3-postpone contradiction"
print(f"\n[OK] seeded Apollo Q2/Q3 contradiction caught")
print(f"cost: ${opus_cost:.6f}  ({opus_in} in / {opus_out} out tokens)  wall: {opus_wall:.2f}s")
for c in opus_contradictions:
    if {c.page_a, c.page_b} == {
        "knowledge/decisions/apollo-q2-ship",
        "knowledge/decisions/apollo-q3-postpone",
    }:
        console.print(Panel(
            f"[bold]Subject:[/] {c.subject}\n"
            f"[bold]Severity:[/] {c.severity}\n"
            f"[bold]Why:[/] {c.why}\n"
            f"[bold]Evidence:[/]\n  - " + "\n  - ".join(c.evidence),
            title="Apollo Q2 vs Q3",
            border_style="yellow",
        ))


In [ ]:
# A/B against Sonnet 4.6.
t0 = time.perf_counter()
sonnet_contradictions, sonnet_in, sonnet_out = detect_contradictions(
    pages, client=client, model="claude-sonnet-4-6"
)
sonnet_wall = time.perf_counter() - t0
sonnet_cost = estimate_cost_usd("claude-sonnet-4-6", sonnet_in, sonnet_out)

t = Table(title="Opus 4.7 vs Sonnet 4.6 — contradiction detection")
t.add_column("model")
t.add_column("contradictions found", justify="right")
t.add_column("caught Apollo seed")
t.add_column("tokens (in/out)", justify="right")
t.add_column("cost (USD)", justify="right")
t.add_column("wall (s)", justify="right")

for label, results, in_tok, out_tok, cost, wall in (
    ("opus-4-7", opus_contradictions, opus_in, opus_out, opus_cost, opus_wall),
    ("sonnet-4-6", sonnet_contradictions, sonnet_in, sonnet_out, sonnet_cost, sonnet_wall),
):
    caught = any(
        {c.page_a, c.page_b} == {
            "knowledge/decisions/apollo-q2-ship",
            "knowledge/decisions/apollo-q3-postpone",
        }
        for c in results
    )
    t.add_row(
        label,
        str(len(results)),
        "yes" if caught else "NO",
        f"{in_tok}/{out_tok}",
        f"${cost:.6f}",
        f"{wall:.2f}",
    )
console.print(t)

print("Rationale receipt: Opus is the right model when the cost of a missed contradiction")
print("(the wiki silently rotting under conflicting state) outweighs the per-call delta.")
print("For high-volume per-page checks, Sonnet would be the call.")


## Part E — Archive flow with atomic backlink rewrite

The §10 Scenario H demo. We pick `apollo-q2-ship` (the contradicted
decision) as the archive target and simulate its upstream Notion source
returning 404 (`archived_reason: upstream-deleted`).

The flow runs entirely under `/tmp/marginalia-nb08-archive/` so the
canonical fixtures stay untouched.

The load-bearing pattern is **plan-then-apply**:

1. `compute_archive_patchset` walks the wiki, builds the full patchset
   in memory, and re-validates every patched page against its
   Pydantic schema. *No disk writes happen here.*
2. `apply_archive_patchset` writes the patchset to disk via
   tmp-then-`os.replace` for crash safety.

A logical failure (a substitution that produces an invalid page) is
caught before any I/O — the dry-run is not just a debugging aid; it's
the same code path apply uses, just stopped before the writes.


In [ ]:
SANDBOX = Path("/tmp/marginalia-nb08-archive/poc-wiki")
copy_wiki_to(WIKI, SANDBOX)
print(f"copied canonical wiki to {SANDBOX}")

target = "knowledge/decisions/apollo-q2-ship"
patchset = compute_archive_patchset(
    SANDBOX,
    target,
    archived_on=TODAY,
    reason=ArchivedReason.UPSTREAM_DELETED,
)

t = Table(title=f"Archive plan for {target}")
t.add_column("field")
t.add_column("value")
t.add_row("old path", str(patchset.old_path.relative_to(SANDBOX)))
t.add_row("new path", str(patchset.new_path.relative_to(SANDBOX)))
t.add_row("old wikilink", patchset.old_wikilink)
t.add_row("new wikilink", patchset.new_wikilink)
t.add_row("target patch changed", str(patchset.target_patch.changed))
t.add_row("referencing pages", str(len(patchset.referencing_patches)))
console.print(t)

ref_t = Table(title="Referencing pages affected")
ref_t.add_column("page")
ref_t.add_column("metadata changed")
ref_t.add_column("body changed")
for rp in patchset.referencing_patches:
    ref_t.add_row(
        str(rp.path.relative_to(SANDBOX)),
        "yes" if rp.old_metadata != rp.new_metadata else "no",
        "yes" if rp.old_body != rp.new_body else "no",
    )
console.print(ref_t)


In [ ]:
# Dry-run: render the unified diff for every changed file.
diff = render_patchset_diff(patchset)
print(diff[:3500])
print("... (truncated)" if len(diff) > 3500 else "")
print()
print(f"Total diff size: {len(diff):,} chars covering {len(patchset.all_patches)} files.")
print("Notice that disk has not been touched yet — dry-run is read-only by construction.")


In [ ]:
# Apply the patchset, then re-walk to verify atomicity.
apply_archive_patchset(patchset)

after = list(walk_wiki(SANDBOX))
print(f"after apply: walked {len(after)} pages")

# 1. Target moved.
moved = next((p for p in after if p.wikilink == "knowledge/decisions/archived/apollo-q2-ship"), None)
assert moved is not None, "target was not moved to archived/"
assert moved.page.status == PageStatus.ARCHIVED
assert moved.page.archived_reason is not None and moved.page.archived_reason.value == "upstream-deleted"
assert moved.page.archived_date == TODAY
print("[OK] target moved to archived/ with status=archived, archived_date, archived_reason set")

# 2. Old path is gone.
assert not (SANDBOX / "knowledge/decisions/apollo-q2-ship.md").exists()
print("[OK] old path deleted")

# 3. No surviving body or related[] references to the old wikilink.
old_marker = "[[knowledge/decisions/apollo-q2-ship]]"
residual = [p for p in after if old_marker in p.body]
assert not residual, f"residual body refs in {[r.path for r in residual]}"
for p in after:
    for field in ("related", "contradicts", "owners"):
        for entry in getattr(p.page, field, []) or []:
            assert entry != old_marker, f"residual {field} entry in {p.path}"
print("[OK] no surviving body or frontmatter references to old wikilink")

# 4. New wikilink IS present where the old one used to be.
new_marker = "[[knowledge/decisions/archived/apollo-q2-ship]]"
referencers = [p.wikilink for p in after if new_marker in p.body]
print(f"[OK] new wikilink {new_marker!r} referenced by: {referencers}")


In [ ]:
# Atomicity test — seed a deliberately-bad patch target.
ATOMIC_SANDBOX = Path("/tmp/marginalia-nb08-archive-atomicity/poc-wiki")
copy_wiki_to(WIKI, ATOMIC_SANDBOX)

# Plant an unrelated page whose `related[]` will become invalid after
# substitution — the strict-schema retry equivalent for the rewriter.
broken = ATOMIC_SANDBOX / "knowledge" / "decisions" / "broken-ref.md"
broken.write_text(
    '---\n'
    'title: Broken Ref\n'
    'type: decision\n'
    'status: active\n'
    'confidence: high\n'
    'created: 2026-04-01\n'
    'last_synced: 2026-04-01\n'
    'owners:\n  - \'[[knowledge/entities/marginalia-engine]]\'\n'
    'supersedes: not-a-wikilink-format\n'
    '---\n'
    'body\n',
    encoding="utf-8",
)

before = sorted(p.relative_to(ATOMIC_SANDBOX) for p in ATOMIC_SANDBOX.rglob("*.md"))
try:
    compute_archive_patchset(
        ATOMIC_SANDBOX,
        "knowledge/decisions/apollo-q2-ship",
        archived_on=TODAY,
        reason=ArchivedReason.UPSTREAM_DELETED,
    )
    raise SystemExit("plan should have raised")
except Exception as exc:
    print(f"[OK] plan failed loud: {type(exc).__name__}")
    print("     -> no disk mutation occurred during the plan phase.")

after = sorted(p.relative_to(ATOMIC_SANDBOX) for p in ATOMIC_SANDBOX.rglob("*.md"))
assert before == after, "a failed plan must not touch disk"
print(f"[OK] disk identical before/after failed plan ({len(after)} files)")


## Part F — `lint-report.md`: aggregating every finding

`lint_wiki()` already composes the three detectors and returns a
`LintReport`. `format_lint_report_md` renders it as the canonical
markdown output. In production this would land at the wiki root; here
we write it to `/tmp/`.


In [ ]:
# Run the full pass once more, this time via the orchestrator function.
report = lint_wiki(WIKI, client=client, today=TODAY, threshold_days=14)
print(f"lint pass: {report.total_pages} pages, {len(report.stale_pages)} stale, "
      f"{len(report.contradictions)} contradictions, {len(report.orphans)} orphans")
print(f"cost: ${report.cost_usd:.6f}  ({report.tokens_in} in / {report.tokens_out} out tokens, {report.wall_seconds:.2f}s)")

REPORT_DIR = Path("/tmp/marginalia-nb08-lint")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
report_md = format_lint_report_md(report, generated_on=TODAY)
report_path = REPORT_DIR / "lint-report.md"
report_path.write_text(report_md, encoding="utf-8")
print(f"\nwrote {report_path} ({len(report_md):,} chars)")

IPyMarkdown(report_md)


## Part G — Receipts + extraction

The cert-valuable artifacts from this notebook live in two places:

- `engine/decisions/lint-and-archive.md` — receipts file with the
  Opus-vs-Sonnet token table, design choices, and re-running snippet.
- The `engine/utils/wiki_walker.py`, `engine/utils/backlink_rewrite.py`,
  `engine/agents/lint/full_pass.py`, and `engine/prompts/lint.md`
  modules — already extracted as the notebook ran (every demo above
  imports from `engine.*`).

The seeded `notebooks/data/poc-wiki/knowledge/` fixtures are tracked
fixtures, not regenerated locally. Future notebooks (NB 09–11) reuse
them.


## What to extract

| Notebook artifact | Extracts to |
|---|---|
| `walk_wiki()` + `WikiPage` dataclass | `engine/utils/wiki_walker.py` (extracted) |
| `compute_archive_patchset()`, `apply_archive_patchset()`, `render_patchset_diff()`, `ArchivePatchSet`, `PagePatch`, `copy_wiki_to`, `all_referenced_wikilinks` | `engine/utils/backlink_rewrite.py` (extracted) |
| `lint_wiki()`, `detect_stale()`, `detect_orphans()`, `detect_contradictions()`, `format_pages_block()`, `format_lint_report_md()`, `LintReport`, `Contradiction`, `StaleFinding`, `OrphanFinding` | `engine/agents/lint/full_pass.py` (extracted) |
| Versioned lint prompt (Opus 4.7, required `<thinking>` CoT) | `engine/prompts/lint.md` (extracted) |
| Cell 11–12 receipts | `engine/decisions/lint-and-archive.md` (extracted) |
| 7 hand-written knowledge pages + `[[wikilinks]]` added to existing source bodies | `notebooks/data/poc-wiki/knowledge/**` (tracked fixtures) |

**Notebook-only (intentionally not extracted):**
- The `rich` tables and `Panel` renders for fixture comparison.
- The dry-run diff print in cell 15 — useful here for visual confirmation, redundant in production where `apply` does the same plan internally.
- The atomicity sandbox in cell 17 — a one-shot demonstration of plan-phase failure containment, not a re-runnable fixture.
- Tmp wiki management under `/tmp/marginalia-nb08-*` — notebook plumbing.

**Out of scope (deferred):**
- **Job-queue invocation of lint** → NB 09. The lint pass is a perfect candidate for a scheduled `marginalia jobs run lint --kind nightly` flow.
- **Caching the contradiction prompt's output** → NB 10. The `CACHE_VERSION` constant lands there; for now, `engine/prompts/lint.md` carries `version: v1` in its frontmatter as the per-prompt anchor.
- **Audit DB rows** → NB 11. Every contradiction surfaced should land an `audit_events` row; the `LintReport` shape already exposes the fields that table will key on.
- **Multi-file archive** (archive N pages atomically) — single-file is sufficient to prove the pattern; N-file is mechanical extension.
- **Rewriting non-`[[path]]` link forms** (raw URLs, relative `./` paths) — design assumes wikilinks are the canonical form; other shapes would need a separate adapter.

**`CACHE_VERSION` discipline:** not yet in place (NB 10). The new `engine/prompts/lint.md` starts at `version: v1`; future edits must bump that and `CACHE_VERSION` together once the cache layer lands.
